# HDB Resale Flat Prices — ETL Pipeline
### Data Engineering Team | Part 1: Developing Data Pipelines

**Scope:** January 2012 – December 2016 resale flat transactions.

**Source:** [data.gov.sg — Resale Flat Prices](https://data.gov.sg/collections/189/view)

This notebook builds an end-to-end ETL pipeline that:
1. **Extracts** raw source files and unions them into a single master dataset (scope-filtered programmatically).
2. **Profiles** the data and applies **data-driven validation rules**.
3. **Cleans**: resolves composite-key duplicates, recomputes remaining lease, flags anomalous prices.
4. **Transforms**: derives the `Resale Identifier` column, resolves identifier-level duplicates.
5. **Hashes** the identifier with SHA-256 (irreversible, uniqueness-preserving).
6. **Outputs** five artefact groups: `raw`, `cleaned`, `failed`, `transformed`, `hashed`.

All reusable logic lives in `../src/*.py` modules (imported below) — this notebook is the
orchestration + documentation layer, per the assignment's emphasis on maintainable,
explainable, and robust engineering.

> **Source data note:** data.gov.sg splits this dataset across several files by date range.
> To cover Jan 2012 – Dec 2016 exactly, I need the **tail end** (Jan–Feb 2012) of the
> "2000 – Feb 2012" file, plus the full "Mar 2012 – Dec 2014" and "Jan 2015 – Dec 2016" files.
> No manually deletion of rows from any file. Hence instead, I load every raw file as is and
> apply a programmatic `month` range filter (see `extract.build_master_dataset`), which
> naturally also excludes the out-of-scope 1990–1999 file without any special-casing. Hope it clarifies the initialisation step I did
> 


## I do the setup first


In [1]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import pandas as pd
import extract, profiling, validate, dedup, lease, anomaly, transform, hashing

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RAW_DIR = "../data/raw"
OUT_DIR = "../outputs"


## 1. Extraction
Load every raw CSV file exactly as downloaded (no manual edits), copy them into
`outputs/raw/`, then union all files' columns and filter programmatically
to the Jan 2012 – Dec 2016 scope. 

In [2]:
raw_frames = extract.load_raw_files(RAW_DIR)
for name, df in raw_frames.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
# Exploratory data analysis 

In [3]:
# Persist raw files as-is into outputs/raw/ (audit trail — untouched copies)
# Copy all downloaded raw files into raw data directory
import shutil
for name in raw_frames:
    shutil.copy(os.path.join(RAW_DIR, name), os.path.join(OUT_DIR, "raw", name))
print("Raw files copied to outputs/raw/")


Raw files copied to outputs/raw/


In [5]:
master = extract.build_master_dataset(raw_frames)
print("Master dataset shape (scope-filtered, Jan 2012 - Dec 2016):", master.shape)
print("Month range:", master['month'].min(), "to", master['month'].max())
print("\nColumns (union of all source files):", master.columns.tolist())
master.head()


ValueError: No objects to concatenate

## 2. Data Profiling
A lightweight profiler (no external dependency) summarising, per column: dtype, null
count/percentage, distinct value count, and either numeric summary stats or top categorical
values. This report grounds the validation rules below in the dataset's actual statistical
properties, as required by the assignment.

In [ ]:
profile_report = profiling.profile_dataframe(master)
profile_report


In [ ]:
profile_report.to_csv(os.path.join(OUT_DIR, "profiling_report.csv"), index=False)
print("Saved profiling report.")


## 3. Data Validation
Validation rules for `month`, `town`, `flat_type`, `flat_model`, and `storey_range` are
derived from the master dataset's own observed values / formats (see `src/validate.py`
docstring for the exact rule for each field), rather than an externally hardcoded list.
We also add two extra sanity rules (`floor_area_sqm` and `resale_price` must be positive
and plausible) as additional data quality checks per the assignment's open-ended requirement.

Rows failing **any** rule are routed to the `failed` output with a reason tag; rows passing
all rules proceed to cleaning.

In [ ]:
validated = validate.run_all_validations(master)

rule_cols = ['valid_month','valid_town','valid_flat_type','valid_flat_model',
             'valid_storey_range','valid_floor_area','valid_resale_price']
print("Validation pass rate:", round(validated['is_valid'].mean() * 100, 2), "%")
for c in rule_cols:
    n_fail = (~validated[c]).sum()
    print(f"  {c}: {n_fail} failing rows")


In [ ]:
valid_rows = validated[validated['is_valid']].drop(columns=rule_cols + ['is_valid']).reset_index(drop=True)
invalid_rows = validated[~validated['is_valid']].copy()

# Tag WHY each invalid row failed (which rule(s))
def failure_reasons(row):
    return "; ".join(c.replace("valid_", "") + "_invalid" for c in rule_cols if not row[c])

if len(invalid_rows):
    invalid_rows['failure_reason'] = invalid_rows.apply(failure_reasons, axis=1)
    invalid_rows = invalid_rows.drop(columns=rule_cols + ['is_valid'])

print("Valid rows proceeding to cleaning:", len(valid_rows))
print("Invalid rows routed to failed/:", len(invalid_rows))


## 4. Cleaning

### 4.1 Composite-key duplicate resolution
Per the assignment, the composite key is *all columns except resale_price*. I additionally
exclude the source-provided `remaining_lease` column from the key, since I discard and
recompute that field in step 4.2 (it is not part of a row's business identity —
see `src/dedup.py` docstring). Where duplicate keys exist, I keep the higher-priced row and
route the rest to `failed`.

In [ ]:
cleaned, dup_failed = dedup.resolve_duplicates(valid_rows)
print("Rows after composite-key dedup (cleaned):", len(cleaned))
print("Rows discarded to failed/ (lower-price duplicates):", len(dup_failed))


### 4.2 Remaining lease recomputation
Assume a 99-year HDB lease. Since `lease_commence_date` only provides a year, we assume the
lease commences 1 January of that year, and compute remaining lease **as of today**, floored
down to whole years + months (see `src/lease.py` docstring for the exact assumption).

In [ ]:
cleaned = lease.add_remaining_lease_columns(cleaned, lease_col="lease_commence_date")
cleaned[['lease_commence_date','remaining_lease_years','remaining_lease_months','remaining_lease_display']].head()


### 4.3 Anomalous resale price detection
**Heuristic:** compute a Z-score for each row's `resale_price` within its peer group
(`town` + `flat_type` + `flat_model`), since price ranges differ enormously across town/type/
model combinations. Rows with `|Z| > 3` (and peer group size ≥ 5, so the standard deviation
is statistically meaningful) are flagged `is_price_anomaly = True`.

**Assumption:** anomalies are flagged for downstream analyst awareness (an extra column), not
silently dropped — the assignment doesn't ask us to remove them, only to identify them.

In [ ]:
cleaned = anomaly.flag_price_anomalies(cleaned)
print("Anomalous rows flagged:", cleaned['is_price_anomaly'].sum(), "/", len(cleaned))
cleaned[cleaned['is_price_anomaly']][['month','town','flat_type','flat_model','resale_price','price_zscore']].head(10)


### 4.4 Save cleaned & failed outputs

In [ ]:
all_failed = pd.concat([invalid_rows, dup_failed], ignore_index=True, sort=False)

cleaned.to_csv(os.path.join(OUT_DIR, "cleaned", "cleaned_resale_flat_prices.csv"), index=False)
all_failed.to_csv(os.path.join(OUT_DIR, "failed", "failed_records.csv"), index=False)

print("Cleaned dataset saved:", cleaned.shape)
print("Failed dataset saved:", all_failed.shape)
all_failed['failure_reason'].value_counts()


## 5. Transformation

### 5.1 Resale Identifier
Derived per the assignment's exact 9-character spec — see `src/transform.py` docstring for
the full breakdown of each character group (block digits, peer-group average price digits,
row's own month, town initial).

In [ ]:
transformed = transform.add_resale_identifier(cleaned)
transformed[['month','town','block','flat_type','resale_price','resale_identifier']].head(10)


In [ ]:
# sanity check: identifier is always exactly 9 characters, starts with 'S'
assert (transformed['resale_identifier'].str.len() == 9).all()
assert (transformed['resale_identifier'].str[0] == 'S').all()
print("Resale Identifier format check passed.")


### 5.2 Resolve identifier-level duplicates
Per the assignment's step 2: if the derived identifier is not unique across rows, keep the
higher-priced row and discard the rest to `failed`.

In [ ]:
transformed, id_dup_failed = transform.resolve_identifier_duplicates(transformed)
print("Rows after identifier dedup (transformed):", len(transformed))
print("Rows discarded to failed/ (identifier duplicates):", len(id_dup_failed))

# append these to the failed dataset too
all_failed = pd.concat([all_failed, id_dup_failed], ignore_index=True, sort=False)
all_failed.to_csv(os.path.join(OUT_DIR, "failed", "failed_records.csv"), index=False)

transformed.to_csv(os.path.join(OUT_DIR, "transformed", "transformed_resale_flat_prices.csv"), index=False)
print("Transformed dataset saved:", transformed.shape)


## 6. Hashing
Hash `resale_identifier` using **SHA-256** — irreversible, and with a 256-bit output space,
collisions across our dataset size are astronomically unlikely, so uniqueness is preserved.
See `src/hashing.py` docstring for the full rationale and its limitations.

In [ ]:
hashed = hashing.add_hashed_identifier(transformed)
print("Unique identifiers:", hashed['resale_identifier'].nunique())
print("Unique hashes:      ", hashed['hashed_identifier'].nunique())
print("Rows:                ", len(hashed))

hashed.to_csv(os.path.join(OUT_DIR, "hashed", "hashed_resale_flat_prices.csv"), index=False)
hashed[['resale_identifier','hashed_identifier']].head()


## 7. Summary

In [ ]:
summary = pd.DataFrame({
    "stage": ["raw (union, scope-filtered)", "valid (passed validation)", "invalid (failed validation)",
              "cleaned (post composite-key dedup)", "failed (composite-key duplicates)",
              "transformed (post identifier dedup)", "failed (identifier duplicates)",
              "hashed (final output)"],
    "row_count": [len(master), len(valid_rows), len(invalid_rows),
                  len(cleaned), len(dup_failed),
                  len(transformed), len(id_dup_failed),
                  len(hashed)]
})
summary


In [ ]:
summary.to_csv(os.path.join(OUT_DIR, "pipeline_summary.csv"), index=False)
print("Pipeline complete. All outputs saved under outputs/{raw,cleaned,failed,transformed,hashed}/")
